<a href="https://colab.research.google.com/github/m-jr-dev/mvp-ml-classificacao/blob/main/notebooks/mvp_classificacao_vinhos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MVP de Classificação de Vinhos

## Objetivo

Este notebook apresenta o processo completo de criação de um modelo de machine learning para classificação de vinhos a partir de atributos químicos. O fluxo contempla carga dos dados por URL, separação entre treino e teste, transformação de dados, treinamento com múltiplos algoritmos clássicos, otimização de hiperparâmetros, avaliação comparativa, exportação do melhor modelo e reflexão final sobre segurança.


## Dataset escolhido

O dataset utilizado é o **Wine Data Set**, disponível publicamente no repositório da UCI. A escolha foi feita porque se trata de um problema clássico de classificação multiclasse com atributos numéricos, adequado para comparar diferentes algoritmos supervisionados.

### URL pública do dataset
`https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data`

### Variável alvo
A coluna `target` representa a classe do vinho.

### Atributos preditores
Os demais campos descrevem propriedades químicas das amostras.


In [1]:
import json
from pathlib import Path

import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier


## Carga do dataset

Nesta etapa, os dados são carregados diretamente pela URL pública. Isso permite que o notebook seja executado no Google Colab do início ao fim sem depender de upload manual de arquivos.


In [2]:
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data"

FEATURE_COLUMNS = [
    "alcohol",
    "malic_acid",
    "ash",
    "alcalinity_of_ash",
    "magnesium",
    "total_phenols",
    "flavanoids",
    "nonflavanoid_phenols",
    "proanthocyanins",
    "color_intensity",
    "hue",
    "od280_od315_of_diluted_wines",
    "proline",
]

TARGET_COLUMN = "target"
COLUMNS = [TARGET_COLUMN, *FEATURE_COLUMNS]

df = pd.read_csv(DATA_URL, header=None, names=COLUMNS)
df.head()


,target,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280_od315_of_diluted_wines,proline
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735


## Inspeção inicial

Antes do treinamento, é importante verificar o formato do dataset, a existência de valores ausentes e a distribuição das classes. Essa inspeção ajuda a confirmar se o problema está apto para o pipeline de classificação.


In [3]:
print(df.shape)
print()
print(df.info())
print()
print(df.isnull().sum())
print()
print(df[TARGET_COLUMN].value_counts().sort_index())


(178, 14)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   target                        178 non-null    int64  
 1   alcohol                       178 non-null    float64
 2   malic_acid                    178 non-null    float64
 3   ash                           178 non-null    float64
 4   alcalinity_of_ash             178 non-null    float64
 5   magnesium                     178 non-null    int64  
 6   total_phenols                 178 non-null    float64
 7   flavanoids                    178 non-null    float64
 8   nonflavanoid_phenols          178 non-null    float64
 9   proanthocyanins               178 non-null    float64
 10  color_intensity               178 non-null    float64
 11  hue                           178 non-null    float64
 12  od280_od315_of_diluted_wines  178 non-null    float64

## Separação entre treino e teste

Foi utilizada a estratégia de **holdout**, reservando 20% dos dados para teste e mantendo a proporção entre as classes com `stratify=y`. Assim, a avaliação final é feita em dados não vistos durante o treinamento.


In [4]:
X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

print("Treino:", X_train.shape, y_train.shape)
print("Teste:", X_test.shape, y_test.shape)


Treino: (142, 13) (142,)
Teste: (36, 13) (36,)


## Transformação dos dados

O enunciado pede tratamento com **normalização** e **padronização**. Para atender a esse ponto, os experimentos consideram os dois tipos de transformação:

- `StandardScaler`: padronização;
- `MinMaxScaler`: normalização.

As transformações são encaixadas dentro de pipelines para evitar vazamento de dados entre treino e teste.


In [5]:
numeric_features = FEATURE_COLUMNS

base_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", "passthrough"),
                ]
            ),
            numeric_features,
        )
    ]
)


## Modelos avaliados

Serão treinados e comparados os quatro algoritmos solicitados:

1. KNN  
2. Árvore de Classificação  
3. Naive Bayes  
4. SVM  

A comparação será feita usando `GridSearchCV` com validação cruzada estratificada e métrica principal `f1_macro`.


In [6]:
search_spaces = {
    "KNN": (
        Pipeline(
            steps=[
                ("preprocessor", base_preprocessor),
                ("classifier", KNeighborsClassifier()),
            ]
        ),
        {
            "preprocessor__numeric__scaler": [StandardScaler(), MinMaxScaler()],
            "classifier__n_neighbors": [3, 5, 7, 9],
            "classifier__weights": ["uniform", "distance"],
            "classifier__metric": ["euclidean", "manhattan"],
        },
    ),
    "Árvore de Decisão": (
        Pipeline(
            steps=[
                ("preprocessor", base_preprocessor),
                ("classifier", DecisionTreeClassifier(random_state=42)),
            ]
        ),
        {
            "preprocessor__numeric__scaler": ["passthrough", StandardScaler(), MinMaxScaler()],
            "classifier__max_depth": [None, 3, 5, 8],
            "classifier__min_samples_split": [2, 4, 6],
            "classifier__criterion": ["gini", "entropy"],
        },
    ),
    "Naive Bayes": (
        Pipeline(
            steps=[
                ("preprocessor", base_preprocessor),
                ("classifier", GaussianNB()),
            ]
        ),
        {
            "preprocessor__numeric__scaler": [StandardScaler(), MinMaxScaler()],
            "classifier__var_smoothing": [1e-09, 1e-08, 1e-07],
        },
    ),
    "SVM": (
        Pipeline(
            steps=[
                ("preprocessor", base_preprocessor),
                ("classifier", SVC(random_state=42)),
            ]
        ),
        {
            "preprocessor__numeric__scaler": [StandardScaler(), MinMaxScaler()],
            "classifier__C": [0.1, 1, 10, 100],
            "classifier__kernel": ["linear", "rbf"],
            "classifier__gamma": ["scale", "auto"],
        },
    ),
}


## Otimização de hiperparâmetros e validação cruzada

Cada algoritmo será ajustado com `GridSearchCV`, usando `StratifiedKFold` com 5 dobras. Essa abordagem permite comparar configurações de forma mais robusta e reduz a chance de escolher um modelo apenas por sorte em uma única divisão dos dados.


In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
best_search = None
best_model_name = None
best_f1 = -1.0

for model_name, (pipeline, params) in search_spaces.items():
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train, y_train)

    predictions = search.predict(X_test)

    result = {
        "model": model_name,
        "best_params": search.best_params_,
        "cv_best_score_f1_macro": round(float(search.best_score_), 4),
        "accuracy": round(float(accuracy_score(y_test, predictions)), 4),
        "precision_macro": round(float(precision_score(y_test, predictions, average="macro")), 4),
        "recall_macro": round(float(recall_score(y_test, predictions, average="macro")), 4),
        "f1_macro": round(float(f1_score(y_test, predictions, average="macro")), 4),
    }
    results.append(result)

    if result["f1_macro"] > best_f1:
        best_f1 = result["f1_macro"]
        best_model_name = model_name
        best_search = search

results_df = pd.DataFrame(results).sort_values(by="f1_macro", ascending=False).reset_index(drop=True)
results_df


,model,best_params,cv_best_score_f1_macro,accuracy,precision_macro,recall_macro,f1_macro
0,KNN,"{'classifier__metric': 'manhattan', 'classifie...",0.9789,1.0000,1.0000,1.0000,1.0000
1,Naive Bayes,"{'classifier__var_smoothing': 1e-09, 'preproce...",0.9727,0.9722,0.9744,0.9762,0.9743
2,Árvore de Decisão,"{'classifier__criterion': 'gini', 'classifier_...",0.9055,0.9444,0.9583,0.9389,0.9457
3,SVM,"{'classifier__C': 10, 'classifier__gamma': 'sc...",0.9927,0.9444,0.9583,0.9333,0.9407


## Comparação dos modelos

A tabela a seguir resume o desempenho dos quatro algoritmos após otimização. O critério principal adotado para seleção foi o `f1_macro`, por equilibrar desempenho entre as classes no cenário multiclasse.


In [8]:
results_df


,model,best_params,cv_best_score_f1_macro,accuracy,precision_macro,recall_macro,f1_macro
0,KNN,"{'classifier__metric': 'manhattan', 'classifie...",0.9789,1.0000,1.0000,1.0000,1.0000
1,Naive Bayes,"{'classifier__var_smoothing': 1e-09, 'preproce...",0.9727,0.9722,0.9744,0.9762,0.9743
2,Árvore de Decisão,"{'classifier__criterion': 'gini', 'classifier_...",0.9055,0.9444,0.9583,0.9389,0.9457
3,SVM,"{'classifier__C': 10, 'classifier__gamma': 'sc...",0.9927,0.9444,0.9583,0.9333,0.9407


## Melhor modelo selecionado

Com base no ranking, o melhor pipeline é separado para inspeção e posterior exportação. Também será exibido um relatório de classificação no conjunto de teste.


In [9]:
best_model_name


'KNN'

In [10]:
best_estimator = best_search.best_estimator_
best_predictions = best_estimator.predict(X_test)

print(classification_report(y_test, best_predictions))


              precision    recall  f1-score   support

           1       1.00      1.00      1.00        12
           2       1.00      1.00      1.00        14
           3       1.00      1.00      1.00        10

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36



## Exportação do modelo resultante

Após a comparação, o melhor modelo é exportado com `joblib`. Também é salvo um arquivo JSON com metadados do experimento, incluindo métricas e limiares de qualidade usados no teste automatizado do back-end.


In [11]:
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

model_path = artifacts_dir / "wine_classifier.joblib"
metadata_path = artifacts_dir / "model_metadata.json"

joblib.dump(best_estimator, model_path)

metadata = {
    "dataset_url": DATA_URL,
    "feature_columns": FEATURE_COLUMNS,
    "target_column": TARGET_COLUMN,
    "selected_model": best_model_name,
    "selected_model_metrics": results_df.iloc[0].to_dict(),
    "ranking": results_df.to_dict(orient="records"),
    "thresholds": {
        "minimum_accuracy": 0.90,
        "minimum_f1_macro": 0.90,
    },
}

metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

print(model_path)
print(metadata_path)


artifacts/wine_classifier.joblib
artifacts/model_metadata.json


## Reflexão sobre segurança

Mesmo em um MVP simples, algumas práticas de segurança são importantes:

- validar todos os campos recebidos pelo back-end;
- coletar apenas os atributos necessários para a predição;
- não expor o arquivo do modelo diretamente ao navegador;
- adotar anonimização ou pseudonimização se o problema envolver dados reais sensíveis;
- registrar erros sem persistir informações sensíveis em logs;
- controlar a implantação do modelo com testes automatizados e versionamento de artefatos.

Esses cuidados ajudam a reduzir risco operacional, vazamento de dados e implantação de modelos com qualidade inadequada.


## Análise final dos resultados

Os experimentos mostraram que diferentes algoritmos respondem de forma distinta ao mesmo conjunto de dados. Como o dataset possui classes relativamente bem separadas, modelos como KNN e SVM tendem a apresentar desempenho elevado quando a transformação dos dados é ajustada corretamente. O uso de validação cruzada e busca de hiperparâmetros foi importante para evitar uma escolha baseada apenas em configuração padrão.

Outro ponto relevante foi o uso combinado de **normalização** e **padronização** dentro de pipelines. Isso garante um fluxo reprodutível, reduz risco de vazamento de dados e deixa a comparação entre modelos mais justa.

Por fim, a exportação do melhor pipeline e a geração de metadados permitem integrar o resultado a uma aplicação full stack simples, além de sustentar um teste automatizado de qualidade no back-end.


## Conclusão

O objetivo do notebook foi cumprido ao construir uma solução completa de classificação com dados carregados por URL pública, pré-processamento adequado, comparação entre KNN, Árvore de Decisão, Naive Bayes e SVM, otimização de hiperparâmetros, avaliação final e exportação do melhor modelo. Esse fluxo atende ao escopo proposto para a parte de machine learning do MVP e prepara o terreno para a integração com a aplicação web.
